# 🔧 F1 2026 Chinese GP — Model Tuning & Evaluation
## Notebook 03

**Insight từ Notebook 02:**
```
Set B (no sprint_pos) Linear Regression: Spearman = 0.544 ← tốt nhất
Set A (with sprint_pos): tất cả models đều kém hơn Set B
→ sprint_pos gây multicollinearity với grid_pos, làm nhiễu model
```

**Pipeline notebook này:**
1. Setup & Load
2. Feature Analysis — tìm hiểu tại sao Set B > Set A
3. Feature Selection — lọc feature tốt nhất
4. Hyperparameter Tuning — Grid Search / LOO-CV
5. Best Model Evaluation
6. Visualization toàn diện
7. Export


## 1. 📦 Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import os, warnings, itertools
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics         import mean_absolute_error, r2_score
from sklearn.pipeline        import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression, RFECV
from scipy.stats             import spearmanr

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

# ── Output ─────────────────────────────────────────────────────
FIG_DIR  = './figures_tuned'
PROC_DIR = './processed'
os.makedirs(FIG_DIR, exist_ok=True)

def savefig(name):
    path = f'{FIG_DIR}/{name}'
    plt.savefig(path, dpi=130, bbox_inches='tight', facecolor='#0f0f0f')
    print(f'  💾 {path}')

# ── Style ─────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':'#0f0f0f', 'axes.facecolor'  :'#1a1a1a',
    'axes.edgecolor'  :'#2e2e2e', 'axes.labelcolor' :'#cccccc',
    'xtick.color'     :'#888888', 'ytick.color'     :'#888888',
    'text.color'      :'#dddddd', 'grid.color'      :'#2a2a2a',
    'grid.linestyle'  :'--',      'font.family'     :'monospace',
    'axes.titlesize'  :12,        'axes.labelsize'  :10,
    'figure.dpi'      :120,
})

TEAM_COLORS = {
    'Mercedes'      :'#00D7B6', 'Ferrari'       :'#E8002D',
    'McLaren'       :'#FF8000', 'Red Bull Racing':'#3671C6',
    'Aston Martin'  :'#229971', 'Alpine'        :'#FF87BC',
    'Williams'      :'#64C4FF', 'Haas F1 Team'  :'#B6BABD',
    'Audi'          :'#52E252', 'Racing Bulls'  :'#6692FF',
    'Cadillac'      :'#C8AA6E',
}

def loo_evaluate(model, X, y):
    """LOO-CV: trả về dict metrics + array predictions."""
    loo  = LeaveOneOut()
    oof  = np.zeros(len(y))
    for tr, te in loo.split(X):
        model.fit(X.iloc[tr], y.iloc[tr])
        oof[te] = model.predict(X.iloc[te])
    mae       = mean_absolute_error(y, oof)
    r2        = r2_score(y, oof)
    sp, p     = spearmanr(y, oof)
    w2        = np.mean(np.abs(y - oof) <= 2)
    w3        = np.mean(np.abs(y - oof) <= 3)
    return {
        'MAE'     : round(mae, 3),
        'R²'      : round(r2,  3),
        'Spearman': round(sp,  3),
        'p-value' : round(p,   4),
        'Within±2': f'{w2:.0%}',
        'Within±3': f'{w3:.0%}',
    }, oof

print(f'✅ Setup done | XGBoost: {HAS_XGB}')
print(f'   Figures → {FIG_DIR}/')


In [ ]:
# ── Load processed data ───────────────────────────────────────
df = pd.read_csv(f'{PROC_DIR}/df_drivers.csv')

for col in ['eliminated_Q1','eliminated_Q2','reached_Q3','dnf','finished']:
    if col in df.columns:
        df[col] = df[col].astype(bool)

driver_team  = dict(zip(df['Driver'], df['Team']))
driver_color = {d: TEAM_COLORS.get(t,'#888888') for d,t in driver_team.items()}

TARGET = 'race_pos'
print(f'✅ Loaded: {df.shape} | n={len(df)} drivers')


## 2. 🔬 Feature Analysis — Tại sao Set B > Set A?

**Hypothesis:** `sprint_pos` có correlation cao với `grid_pos` → multicollinearity  
→ Linear model bị overfit/unstable khi cả hai cùng có mặt


In [ ]:
# ── 2.1 Correlation sprint_pos vs grid_pos & race_pos ─────────
check_cols = ['grid_pos','sprint_pos','sprint_pos_gained',
              'quali_gap','sq_gap','race_pos']
check_df   = df[check_cols].dropna()

corr_m = check_df.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0f0f0f')

# Heatmap
sns.heatmap(corr_m, ax=axes[0], cmap='coolwarm', center=0,
            annot=True, fmt='.3f', linewidths=0.4, linecolor='#0f0f0f',
            cbar_kws={'shrink':0.8})
axes[0].set_title('Correlation — Key Features', color='white', pad=10)
axes[0].tick_params(colors='white')

# Scatter: sprint_pos vs grid_pos
ax = axes[1]
valid = df[['Driver','Team','sprint_pos','grid_pos','race_pos']].dropna()
for _, row in valid.iterrows():
    c = TEAM_COLORS.get(row['Team'],'#888888')
    ax.scatter(row['grid_pos'], row['sprint_pos'], color=c, s=80, alpha=0.9, zorder=5)
    ax.annotate(row['Driver'], (row['grid_pos'], row['sprint_pos']),
                textcoords='offset points', xytext=(5,4), fontsize=7.5, color='white')
z  = np.polyfit(valid['grid_pos'], valid['sprint_pos'], 1)
xl = np.linspace(1, 22, 100)
ax.plot(xl, np.poly1d(z)(xl), '--', color='white', linewidth=1.5, alpha=0.7)
r_gs = valid[['grid_pos','sprint_pos']].corr().iloc[0,1]
ax.set_title(f'Grid Position vs Sprint Position\nr = {r_gs:.3f}',
             color='white', pad=10)
ax.set_xlabel('Grid Position'); ax.set_ylabel('Sprint Position')
ax.invert_xaxis(); ax.invert_yaxis(); ax.grid(alpha=0.2)

plt.suptitle('Why sprint_pos hurts Set A models',
             color='white', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); savefig('01_sprint_pos_analysis.png'); plt.show()

r_sg = valid[['sprint_pos','grid_pos']].corr().iloc[0,1]
r_sr = valid[['sprint_pos','race_pos']].corr().iloc[0,1]
r_gr = df[['grid_pos','race_pos']].corr().iloc[0,1]
print(f'sprint_pos ↔ grid_pos  : r = {r_sg:.3f}  ← high collinearity!')
print(f'sprint_pos ↔ race_pos  : r = {r_sr:.3f}')
print(f'grid_pos   ↔ race_pos  : r = {r_gr:.3f}')
print()
print('→ Khi sprint_pos và grid_pos đều có mặt, linear model bị confused')
print('  vì chúng mang thông tin gần như giống nhau.')


In [ ]:
# ── 2.2 Variance Inflation Factor (VIF) ──────────────────────
# VIF > 5 = có vấn đề multicollinearity
from numpy.linalg import lstsq

def compute_vif(X_df):
    """Tính VIF cho từng feature."""
    vif_data = {}
    X_arr = X_df.values
    for i, col in enumerate(X_df.columns):
        y_i   = X_arr[:, i]
        X_rest= np.delete(X_arr, i, axis=1)
        # Add intercept
        X_int = np.column_stack([np.ones(len(X_rest)), X_rest])
        coef, _, _, _ = lstsq(X_int, y_i, rcond=None)
        y_hat = X_int @ coef
        ss_res= np.sum((y_i - y_hat)**2)
        ss_tot= np.sum((y_i - y_i.mean())**2)
        r2    = 1 - ss_res/ss_tot if ss_tot > 0 else 0
        vif   = 1/(1 - r2) if r2 < 1 else np.inf
        vif_data[col] = round(vif, 2)
    return pd.Series(vif_data).sort_values(ascending=False)

# Tính VIF cho subset features quan trọng
vif_cols = ['grid_pos','sprint_pos','sprint_pos_gained',
            'quali_gap','sq_gap','quali_laptime','sq_laptime']
vif_df   = df[vif_cols].dropna()
vif_series = compute_vif(StandardScaler().fit_transform(
                pd.DataFrame(vif_df, columns=vif_cols))
             if False else vif_df)

# Simplified: dùng corr matrix để tính VIF proxy
print('VIF (Variance Inflation Factor):')
print('  > 10 = severe multicollinearity')
print('  5-10 = moderate')
print('  < 5  = OK')
print()
print(vif_series.to_string())

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0f0f0f')
colors = ['#ff1744' if v>10 else '#ffaa00' if v>5 else '#00e676'
          for v in vif_series.values]
ax.bar(range(len(vif_series)), vif_series.values, color=colors, alpha=0.85)
ax.axhline(5,  color='#ffaa00', linewidth=1.5, linestyle='--', label='VIF=5')
ax.axhline(10, color='#ff1744', linewidth=1.5, linestyle='--', label='VIF=10')
ax.set_xticks(range(len(vif_series)))
ax.set_xticklabels(vif_series.index, rotation=25, ha='right')
ax.set_title('Variance Inflation Factor — Key Features',
             color='white', pad=10)
ax.legend(facecolor='#1a1a1a', labelcolor='white')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout(); savefig('02_vif.png'); plt.show()


## 3. 🎯 Feature Selection

**Strategy:**
1. **Remove high-VIF features** — giữ 1 trong cặp (grid_pos, sprint_pos)
2. **SelectKBest** — chọn top-k features theo F-statistic
3. **Manual curated set** — dựa trên domain knowledge + correlation


In [ ]:
# ── Rebuild feature table với engineered features ─────────────
df_feat = df.copy()

# Engineered (copy từ notebook 02)
df_feat['sq_theoretical_best']     = df_feat['sq_s1']     + df_feat['sq_s2']     + df_feat['sq_s3']
df_feat['sprint_theoretical_best'] = df_feat['sprint_s1'] + df_feat['sprint_s2'] + df_feat['sprint_s3']
df_feat['quali_theoretical_best']  = df_feat['quali_s1']  + df_feat['quali_s2']  + df_feat['quali_s3']
df_feat['sq_time_loss']     = df_feat['sq_laptime']     - df_feat['sq_theoretical_best']
df_feat['sprint_time_loss'] = df_feat['sprint_laptime'] - df_feat['sprint_theoretical_best']
df_feat['quali_time_loss']  = df_feat['quali_laptime']  - df_feat['quali_theoretical_best']
df_feat['sq_to_quali_speed']     = df_feat['quali_speed']  - df_feat['sq_speed']
df_feat['sprint_to_quali_speed'] = df_feat['quali_speed']  - df_feat['sprint_speed']
df_feat['sq_rank']      = df_feat['sq_laptime'].rank()
df_feat['sprint_rank']  = df_feat['sprint_laptime'].rank()
df_feat['quali_rank']   = df_feat['quali_laptime'].rank()
df_feat['rank_change_sq_to_quali']      = df_feat['quali_rank'] - df_feat['sq_rank']
df_feat['rank_change_sprint_to_quali']  = df_feat['quali_rank'] - df_feat['sprint_rank']
df_feat['q_segment'] = (df_feat['reached_Q3'].astype(int)*2 +
                        df_feat['eliminated_Q2'].astype(int))
for s in ['sq_s1','sq_s2','sq_s3','quali_s1','quali_s2','quali_s3']:
    df_feat[f'{s}_vs_field'] = df_feat[s] - df_feat[s].median()

# ── 3 Feature Sets để so sánh ─────────────────────────────────

# Set C: curated — dựa trên Set B nhưng trim collinear features
SET_C = [
    'sq_gap', 'sq_s1', 'sq_s2', 'sq_s3', 'sq_speed',
    'sq_sector_std', 'sq_time_loss',
    'sprint_laptime', 'sprint_gap', 'sprint_speed', 'sprint_sector_std',
    'quali_gap', 'quali_s1', 'quali_s2', 'quali_s3', 'quali_speed',
    'quali_time_loss', 'q_segment',
    'fp1_to_sq_delta', 'sq_to_quali_delta', 'quali_vs_sprint',
    'rank_change_sq_to_quali',
    'grid_pos',   # giữ grid_pos, drop sprint_pos
]

# Set D: minimal — top correlators only
SET_D = [
    'quali_gap', 'grid_pos', 'sq_gap',
    'sprint_laptime', 'quali_laptime',
    'rank_change_sq_to_quali', 'sq_to_quali_delta',
    'quali_s1', 'quali_s2', 'quali_s3',
    'q_segment',
]

# Set E: Set D + sprint_pos_gained (bỏ sprint_pos, giữ gained)
SET_E = SET_D + ['sprint_pos_gained']

print('Feature sets:')
print(f'  Set C (curated, no sprint_pos): {len(SET_C)} features')
print(f'  Set D (minimal):                {len(SET_D)} features')
print(f'  Set E (D + sprint_pos_gained):  {len(SET_E)} features')


## 4. ⚙️ Hyperparameter Tuning

Dùng **LOO-CV làm inner loop** thay vì GridSearchCV  
(vì n=21 quá nhỏ cho nested CV).

Grid search thủ công trên các hyperparameters quan trọng nhất.


In [ ]:
# ── 4.1 Chuẩn bị data cho từng set ───────────────────────────
def make_Xy(feat_cols, df_source):
    valid = df_source.dropna(subset=feat_cols + [TARGET]).reset_index(drop=True)
    valid[TARGET] = valid[TARGET].astype(int)
    X = valid[feat_cols].copy()
    y = valid[TARGET].copy()
    sc = StandardScaler()
    Xs = pd.DataFrame(sc.fit_transform(X), columns=X.columns)
    return valid, X, Xs, y, sc

ml_C, X_C, Xs_C, y_C, sc_C = make_Xy(SET_C, df_feat)
ml_D, X_D, Xs_D, y_D, sc_D = make_Xy(SET_D, df_feat)
ml_E, X_E, Xs_E, y_E, sc_E = make_Xy(SET_E, df_feat)

print(f'Set C: {Xs_C.shape} | Set D: {Xs_D.shape} | Set E: {Xs_E.shape}')
print(f'n drivers: C={len(y_C)}, D={len(y_D)}, E={len(y_E)}')


In [ ]:
# ── 4.2 Ridge alpha tuning ────────────────────────────────────
print('Ridge alpha tuning (Set D — minimal, cleanest)...\n')

alphas = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]
ridge_results = {}

for alpha in alphas:
    m = Ridge(alpha=alpha)
    metrics, _ = loo_evaluate(m, Xs_D, y_D)
    ridge_results[alpha] = metrics
    print(f'  α={alpha:6.2f} | MAE={metrics["MAE"]:.3f} | '
          f'Spearman={metrics["Spearman"]:.3f} | ±2={metrics["Within±2"]}')

best_alpha = max(ridge_results, key=lambda a: ridge_results[a]['Spearman'])
print(f'\n✅ Best α: {best_alpha} → Spearman={ridge_results[best_alpha]["Spearman"]}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0f0f0f')
sp_vals  = [ridge_results[a]['Spearman'] for a in alphas]
mae_vals = [ridge_results[a]['MAE']      for a in alphas]
for ax, vals, title, color, note in [
    (axes[0], sp_vals,  'Spearman r vs α', '#00e676', '↑ better'),
    (axes[1], mae_vals, 'MAE vs α',        '#ff4444', '↓ better'),
]:
    ax.plot(range(len(alphas)), vals, color=color, linewidth=2, marker='o', markersize=7)
    best_i = vals.index(max(vals)) if note=='↑ better' else vals.index(min(vals))
    ax.scatter([best_i], [vals[best_i]], color='#ffd700', s=200, zorder=6,
               label=f'Best α={alphas[best_i]}')
    ax.set_xticks(range(len(alphas)))
    ax.set_xticklabels([str(a) for a in alphas], rotation=30)
    ax.set_xlabel('Alpha'); ax.set_title(f'{title}  ({note})', color='white', pad=8)
    ax.legend(facecolor='#1a1a1a', labelcolor='white')
    ax.grid(alpha=0.25)
plt.suptitle('Ridge Regularization Tuning', color='white', fontsize=13,
             fontweight='bold', y=1.02)
plt.tight_layout(); savefig('03_ridge_alpha_tuning.png'); plt.show()


In [ ]:
# ── 4.3 Random Forest tuning ─────────────────────────────────
print('Random Forest tuning (Set C)...\n')

rf_grid = {
    'n_estimators' : [100, 200, 500],
    'max_depth'    : [2, 3, 4, None],
    'min_samples_leaf': [1, 2, 3],
}

rf_results = {}
total = len(rf_grid['n_estimators'])*len(rf_grid['max_depth'])*len(rf_grid['min_samples_leaf'])
count = 0

best_rf_spear = -999
best_rf_params = {}

for n_est in rf_grid['n_estimators']:
    for depth in rf_grid['max_depth']:
        for leaf in rf_grid['min_samples_leaf']:
            m = RandomForestRegressor(n_estimators=n_est, max_depth=depth,
                                      min_samples_leaf=leaf, random_state=42,
                                      n_jobs=-1)
            metrics, _ = loo_evaluate(m, Xs_C, y_C)
            key = f'n={n_est},d={depth},l={leaf}'
            rf_results[key] = metrics
            count += 1
            if metrics['Spearman'] > best_rf_spear:
                best_rf_spear  = metrics['Spearman']
                best_rf_params = {'n_estimators':n_est,'max_depth':depth,
                                  'min_samples_leaf':leaf}
                print(f'  ✨ New best: {key} → Spearman={metrics["Spearman"]:.3f} | MAE={metrics["MAE"]:.3f}')

print(f'\n✅ Best RF params: {best_rf_params}')
print(f'   Spearman={best_rf_spear:.3f}')


In [ ]:
# ── 4.4 GradientBoosting tuning ──────────────────────────────
print('Gradient Boosting tuning (Set D)...\n')

gb_grid = {
    'n_estimators' : [50, 100, 200],
    'max_depth'    : [2, 3],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample'    : [0.8, 1.0],
}

best_gb_spear  = -999
best_gb_params = {}

for n_est in gb_grid['n_estimators']:
    for depth in gb_grid['max_depth']:
        for lr in gb_grid['learning_rate']:
            for sub in gb_grid['subsample']:
                m = GradientBoostingRegressor(
                    n_estimators=n_est, max_depth=depth,
                    learning_rate=lr, subsample=sub, random_state=42)
                metrics, _ = loo_evaluate(m, Xs_D, y_D)
                if metrics['Spearman'] > best_gb_spear:
                    best_gb_spear  = metrics['Spearman']
                    best_gb_params = {'n_estimators':n_est,'max_depth':depth,
                                      'learning_rate':lr,'subsample':sub}
                    print(f'  ✨ New best: n={n_est},d={depth},lr={lr},sub={sub} '
                          f'→ Spearman={metrics["Spearman"]:.3f} | MAE={metrics["MAE"]:.3f}')

print(f'\n✅ Best GB params: {best_gb_params}')
print(f'   Spearman={best_gb_spear:.3f}')


In [ ]:
# ── 4.5 XGBoost tuning (nếu có) ──────────────────────────────
best_xgb_params = {}
best_xgb_spear  = -999

if HAS_XGB:
    print('XGBoost tuning (Set D)...\n')
    xgb_grid = {
        'n_estimators' : [50, 100, 200],
        'max_depth'    : [2, 3, 4],
        'learning_rate': [0.05, 0.1, 0.2],
        'subsample'    : [0.8, 1.0],
        'reg_alpha'    : [0, 0.1, 1.0],
    }
    for n_est in xgb_grid['n_estimators']:
        for depth in xgb_grid['max_depth']:
            for lr in xgb_grid['learning_rate']:
                for sub in xgb_grid['subsample']:
                    for reg_a in xgb_grid['reg_alpha']:
                        m = XGBRegressor(
                            n_estimators=n_est, max_depth=depth,
                            learning_rate=lr, subsample=sub,
                            reg_alpha=reg_a, random_state=42, verbosity=0)
                        metrics, _ = loo_evaluate(m, Xs_D, y_D)
                        if metrics['Spearman'] > best_xgb_spear:
                            best_xgb_spear  = metrics['Spearman']
                            best_xgb_params = {
                                'n_estimators':n_est,'max_depth':depth,
                                'learning_rate':lr,'subsample':sub,'reg_alpha':reg_a}
                            print(f'  ✨ New best: {best_xgb_params} '
                                  f'→ Spearman={metrics["Spearman"]:.3f}')
    print(f'\n✅ Best XGB params: {best_xgb_params}')
else:
    print('XGBoost not available — skipping')


## 5. 📊 Final Model Evaluation

In [ ]:
# ── Build final tuned models ──────────────────────────────────
final_models = {
    f'Ridge (α={best_alpha}) [D]': (
        Ridge(alpha=best_alpha), Xs_D, y_D, ml_D
    ),
    f'RF tuned [C]': (
        RandomForestRegressor(**best_rf_params, random_state=42, n_jobs=-1),
        Xs_C, y_C, ml_C
    ),
    f'GB tuned [D]': (
        GradientBoostingRegressor(**best_gb_params, random_state=42),
        Xs_D, y_D, ml_D
    ),
    # Baseline để so sánh
    'Linear Reg [D] (baseline)': (
        LinearRegression(), Xs_D, y_D, ml_D
    ),
}

if HAS_XGB and best_xgb_params:
    final_models[f'XGB tuned [D]'] = (
        XGBRegressor(**best_xgb_params, random_state=42, verbosity=0),
        Xs_D, y_D, ml_D
    )

final_results = {}
final_preds   = {}

print('Final evaluation (LOO-CV):\n')
print(f'{"Model":35s} | MAE   | R²     | Spearman | ±2    | ±3')
print('─'*75)

for name, (model, Xs, y_use, ml_use) in final_models.items():
    metrics, oof = loo_evaluate(model, Xs, y_use)
    final_results[name] = {**metrics, 'n': len(y_use)}
    final_preds[name]   = (oof, y_use, ml_use)
    # Retrain full
    model.fit(Xs, y_use)
    print(f'  {name:35s} | {metrics["MAE"]:.3f} | {metrics["R²"]:.3f}  | '
          f'{metrics["Spearman"]:.3f}    | {metrics["Within±2"]}   | {metrics["Within±3"]}')

# Best overall
best_final = max(final_results, key=lambda k: final_results[k]['Spearman'])
print(f'\n🏆 Best: {best_final}')
print(f'   {final_results[best_final]}')


In [ ]:
# ── Comparison chart: baseline vs tuned ──────────────────────
names     = list(final_results.keys())
mae_vals  = [final_results[n]['MAE']      for n in names]
spear_vals= [final_results[n]['Spearman'] for n in names]
w2_vals   = [int(final_results[n]['Within±2'].replace('%','')) for n in names]

fig, axes = plt.subplots(1, 3, figsize=(17, 6))
fig.patch.set_facecolor('#0f0f0f')

for ax, vals, title, note, lower in [
    (axes[0], mae_vals,   'MAE',        '↓ better', True),
    (axes[1], spear_vals, 'Spearman r', '↑ better', False),
    (axes[2], w2_vals,    'Within ±2 (%)', '↑ better', False),
]:
    best_i = vals.index(min(vals)) if lower else vals.index(max(vals))
    colors = ['#ffd700' if i==best_i else '#4a9eff' for i in range(len(vals))]
    # Baseline dùng màu xám
    colors = ['#888888' if 'baseline' in n else c
              for n, c in zip(names, colors)]
    ax.bar(range(len(vals)), vals, color=colors, alpha=0.85)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels([n.replace(' [','\n[') for n in names],
                       rotation=20, ha='right', fontsize=8)
    ax.set_title(f'{title}  ({note})', color='white', fontsize=11, pad=8)
    ax.grid(axis='y', alpha=0.25)
    for i,v in enumerate(vals):
        ax.text(i, v+max(vals)*0.015, f'{v:.0f}%' if title=='Within ±2 (%)' else f'{v:.3f}',
                ha='center', fontsize=8, color='white')

legend_items = [
    mpatches.Patch(color='#ffd700', label='Best model'),
    mpatches.Patch(color='#4a9eff', label='Tuned'),
    mpatches.Patch(color='#888888', label='Baseline'),
]
fig.legend(handles=legend_items, loc='upper right',
           facecolor='#1a1a1a', labelcolor='white', fontsize=9)

plt.suptitle('Tuned Models vs Baseline — LOO-CV Evaluation',
             color='white', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); savefig('04_final_model_comparison.png'); plt.show()


## 6. 📈 Visualization — Best Model

In [ ]:
best_preds_arr, best_y, best_ml = final_preds[best_final]
best_model_obj = final_models[best_final][0]

error_df = pd.DataFrame({
    'Driver'   : best_ml['Driver'].values,
    'Team'     : best_ml['Team'].values,
    'Grid'     : best_ml['grid_pos'].values,
    'Sprint'   : best_ml.get('sprint_pos', pd.Series([np.nan]*len(best_ml))).values
                 if 'sprint_pos' in best_ml.columns else np.nan,
    'Actual'   : best_y.values,
    'Predicted': best_preds_arr.round(1),
    'Error'    : (best_preds_arr - best_y.values).round(2),
    'AbsError' : np.abs(best_preds_arr - best_y.values).round(2),
}).sort_values('Actual').reset_index(drop=True)

# ── 6.1 Actual vs Predicted ───────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor('#0f0f0f')

for i, row in error_df.iterrows():
    c = TEAM_COLORS.get(row['Team'],'#888888')
    m = 'x' if best_ml['dnf'].iloc[
        best_ml['Driver'].values.tolist().index(row['Driver'])] else 'o'
    size = 160 if row['AbsError']>4 else 100
    ax.scatter(row['Actual'], row['Predicted'], color=c, s=size, marker=m,
               alpha=0.9, zorder=5)
    ax.annotate(row['Driver'], (row['Actual'], row['Predicted']),
                textcoords='offset points', xytext=(7,5), fontsize=8.5, color='white')

ax.plot([1,22],[1,22],'--',color='#444',linewidth=1.5,label='Perfect prediction')
ax.fill_between([1,22],[max(1,l-2) for l in [1,22]],
                       [min(22,l+2) for l in [1,22]],
                alpha=0.08, color='#4a9eff', label='±2 positions')
ax.fill_between([1,22],[max(1,l-4) for l in [1,22]],
                       [min(22,l+4) for l in [1,22]],
                alpha=0.04, color='#ffaa00', label='±4 positions')

r = final_results[best_final]
ax.set_title(f'{best_final}\nMAE={r["MAE"]} pos | Spearman={r["Spearman"]} | '
             f'Within±2={r["Within±2"]} | Within±3={r["Within±3"]}',
             color='white', fontsize=11, pad=12)
ax.set_xlabel('Actual Race Position', fontsize=11)
ax.set_ylabel('Predicted Race Position', fontsize=11)
ax.legend(facecolor='#1a1a1a', labelcolor='white')
ax.grid(alpha=0.2)
ax.set_xlim(0,23); ax.set_ylim(-2,26)

legend_items = [mpatches.Patch(color=c,label=t)
                for t,c in TEAM_COLORS.items() if t in driver_team.values()]
ax2_leg = ax.twinx()
ax2_leg.set_yticks([])
ax2_leg.legend(handles=legend_items, loc='lower right',
               facecolor='#1a1a1a', labelcolor='white', fontsize=7.5, ncol=2)

plt.tight_layout(); savefig('05_actual_vs_predicted.png'); plt.show()


In [ ]:
# ── 6.2 Error breakdown ───────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.patch.set_facecolor('#0f0f0f')

# Top: error per driver với label Grid→Sprint→Race
ax = axes[0]
bar_c = ['#00e676' if abs(e)<=2 else '#ffaa00' if abs(e)<=4 else '#ff1744'
         for e in error_df['Error']]
ax.bar(range(len(error_df)), error_df['Error'], color=bar_c, alpha=0.85)
ax.axhline(0, color='white', linewidth=1)
ax.set_xticks(range(len(error_df)))

def make_label(row):
    g = f'G{int(row.Grid)}' if pd.notna(row.Grid) else 'G?'
    s = f'S{int(row.Sprint)}' if pd.notna(row.Sprint) else ''
    r = f'R{int(row.Actual)}'
    return f'{row.Driver}\n{g}{"→"+s if s else ""}→{r}'

ax.set_xticklabels([make_label(r) for _,r in error_df.iterrows()],
                   fontsize=7.5, rotation=45, ha='right')
ax.set_ylabel('Error (Predicted − Actual)')
ax.set_title(f'Prediction Error — {best_final}\n🟢≤±2  🟡≤±4  🔴>±4',
             color='white', pad=10)
ax.grid(axis='y', alpha=0.25)

# Bottom: error histogram + cumulative %
ax2 = axes[1]
abs_errs = error_df['AbsError'].values
bins = np.arange(0, int(abs_errs.max())+2, 1)
counts, edges = np.histogram(abs_errs, bins=bins)
ax2.bar(edges[:-1], counts, width=0.85, color='#4a9eff', alpha=0.8, label='Count')
ax2.set_xlabel('Absolute Error (positions)')
ax2.set_ylabel('Number of Drivers', color='#4a9eff')
ax2.grid(axis='y', alpha=0.25)

ax2r = ax2.twinx()
cumulative = np.cumsum(counts) / len(abs_errs) * 100
ax2r.step(edges[:-1], cumulative, color='#ffd700', linewidth=2.5,
          where='post', label='Cumulative %')
ax2r.axhline(50, color='#888', linewidth=1, linestyle=':')
ax2r.axhline(80, color='#888', linewidth=1, linestyle=':')
ax2r.set_ylabel('Cumulative %', color='#ffd700')
ax2r.set_ylim(0, 110)

lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2r.get_legend_handles_labels()
ax2.legend(lines1+lines2, labels1+labels2,
           facecolor='#1a1a1a', labelcolor='white', fontsize=9)

w2  = int(final_results[best_final]['Within±2'].replace('%',''))
w3  = int(final_results[best_final]['Within±3'].replace('%',''))
ax2.set_title(f'Error Distribution — Within±2: {w2}%  |  Within±3: {w3}%',
              color='white', pad=8)

plt.suptitle('Prediction Error Analysis',
             color='white', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); savefig('06_error_analysis.png'); plt.show()


In [ ]:
# ── 6.3 Feature importance (best tree model) ─────────────────
tree_key = [k for k in final_results if 'RF' in k or 'GB' in k or 'XGB' in k]
if tree_key:
    tree_key = max(tree_key, key=lambda k: final_results[k]['Spearman'])
    tree_model, tree_Xs, tree_y, tree_ml = final_models[tree_key]
    tree_model.fit(tree_Xs, tree_y)
    feat_cols_used = tree_Xs.columns.tolist()
    imp = pd.Series(tree_model.feature_importances_, index=feat_cols_used)\
            .sort_values(ascending=True)

    def feat_color(f):
        if f.startswith('fp1'):    return '#88ccff'
        if f.startswith('sq'):     return '#4a9eff'
        if f.startswith('sprint'): return '#ff9900'
        if f.startswith('quali'):  return '#ffaa00'
        if f == 'grid_pos':        return '#ff4444'
        return '#aa88ff'

    fig, ax = plt.subplots(figsize=(10, 9))
    fig.patch.set_facecolor('#0f0f0f')
    ax.barh(range(len(imp)), imp.values,
            color=[feat_color(f) for f in imp.index], alpha=0.85)
    ax.set_yticks(range(len(imp)))
    ax.set_yticklabels(imp.index, fontsize=8.5)
    ax.set_title(f'{tree_key} — Feature Importance', color='white', pad=12)
    ax.set_xlabel('Importance Score')
    ax.grid(axis='x', alpha=0.25)
    for rank, (feat, val) in enumerate(imp.sort_values(ascending=False).head(5).items()):
        idx = list(imp.index).index(feat)
        ax.text(val+0.003, idx, f'#{rank+1}', va='center', fontsize=9, color='white')

    legend_items = [
        mpatches.Patch(color='#88ccff', label='FP1'),
        mpatches.Patch(color='#4a9eff', label='Sprint Qualifying'),
        mpatches.Patch(color='#ff9900', label='Sprint'),
        mpatches.Patch(color='#ffaa00', label='Qualifying'),
        mpatches.Patch(color='#ff4444', label='Grid'),
        mpatches.Patch(color='#aa88ff', label='Engineered'),
    ]
    ax.legend(handles=legend_items, loc='lower right',
              facecolor='#1a1a1a', labelcolor='white', fontsize=9)
    plt.tight_layout(); savefig('07_feature_importance.png'); plt.show()


In [ ]:
# ── 6.4 Learning curve (LOO error vs feature count) ──────────
print('Tính learning curve — LOO error vs số features...')

all_feat_pool = SET_C  # dùng Set C để có đủ features
feat_pool_df  = df_feat.dropna(subset=all_feat_pool+[TARGET]).reset_index(drop=True)
feat_pool_df[TARGET] = feat_pool_df[TARGET].astype(int)

# Sort features theo correlation
corr_all = feat_pool_df[all_feat_pool+[TARGET]].corr()[TARGET].drop(TARGET)\
    .abs().sort_values(ascending=False)
sorted_feats = corr_all.index.tolist()

lc_results = []
for k in range(1, len(sorted_feats)+1):
    feats_k = sorted_feats[:k]
    Xk      = feat_pool_df[feats_k]
    yk      = feat_pool_df[TARGET]
    Xk_sc   = pd.DataFrame(StandardScaler().fit_transform(Xk), columns=feats_k)
    m       = Ridge(alpha=best_alpha)
    met, _  = loo_evaluate(m, Xk_sc, yk)
    lc_results.append({'k':k, 'feat':feats_k[-1],
                       'MAE':met['MAE'], 'Spearman':met['Spearman']})

lc_df = pd.DataFrame(lc_results)
best_k = lc_df.loc[lc_df['Spearman'].idxmax(), 'k']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f0f')

axes[0].plot(lc_df['k'], lc_df['Spearman'], color='#00e676', linewidth=2)
axes[0].axvline(best_k, color='#ffd700', linewidth=1.5, linestyle='--',
                label=f'Best k={best_k}')
axes[0].set_xlabel('Number of Features')
axes[0].set_ylabel('Spearman r')
axes[0].set_title('Spearman vs Feature Count', color='white', pad=8)
axes[0].legend(facecolor='#1a1a1a', labelcolor='white')
axes[0].grid(alpha=0.25)

axes[1].plot(lc_df['k'], lc_df['MAE'], color='#ff4444', linewidth=2)
axes[1].axvline(best_k, color='#ffd700', linewidth=1.5, linestyle='--',
                label=f'Best k={best_k}')
axes[1].set_xlabel('Number of Features')
axes[1].set_ylabel('MAE')
axes[1].set_title('MAE vs Feature Count', color='white', pad=8)
axes[1].legend(facecolor='#1a1a1a', labelcolor='white')
axes[1].grid(alpha=0.25)

plt.suptitle('Learning Curve — Ridge (α tuned) on Set C',
             color='white', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); savefig('08_learning_curve.png'); plt.show()

print(f'\nOptimal feature count: k={best_k}')
print(f'Top {best_k} features: {sorted_feats[:best_k]}')


## 7. 💡 Summary & Export

In [ ]:
# ── Summary dashboard ─────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#0f0f0f')
fig.text(0.5, 0.98, '🏎️  F1 2026 CHINESE GP — TUNED MODEL DASHBOARD',
         ha='center', va='top', color='white', fontsize=16, fontweight='bold')
r_best = final_results[best_final]
fig.text(0.5, 0.955,
         f'Best: {best_final}  |  MAE={r_best["MAE"]}  |  '
         f'Spearman={r_best["Spearman"]}  |  ±2={r_best["Within±2"]}  |  ±3={r_best["Within±3"]}',
         ha='center', va='top', color='#aaaaaa', fontsize=11)

gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35,
                      top=0.90, bottom=0.08, left=0.06, right=0.97)

# [0,0] Model comparison Spearman
ax0 = fig.add_subplot(gs[0,0])
sp_f  = [final_results[n]['Spearman'] for n in names]
cols  = ['#ffd700' if n==best_final else '#888888' if 'baseline' in n else '#4a9eff'
         for n in names]
ax0.barh(range(len(names)), sp_f, color=cols, alpha=0.85)
ax0.set_yticks(range(len(names)))
ax0.set_yticklabels([n.replace(' [','\n[') for n in names], fontsize=7.5)
ax0.set_title('Spearman r (↑ better)', color='white', fontsize=10)
ax0.grid(axis='x', alpha=0.25)
for i,v in enumerate(sp_f):
    ax0.text(v+0.005, i, f'{v:.3f}', va='center', fontsize=8, color='white')

# [0,1] Ridge alpha curve
ax1 = fig.add_subplot(gs[0,1])
ax1.plot(range(len(alphas)), sp_vals, color='#00e676', linewidth=2, marker='o', markersize=6)
best_ai = sp_vals.index(max(sp_vals))
ax1.scatter([best_ai],[sp_vals[best_ai]], color='#ffd700', s=150, zorder=6,
            label=f'Best α={alphas[best_ai]}')
ax1.set_xticks(range(len(alphas)))
ax1.set_xticklabels([str(a) for a in alphas], rotation=30, fontsize=8)
ax1.set_title('Ridge α Tuning', color='white', fontsize=10)
ax1.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)
ax1.grid(alpha=0.25)

# [0,2] Error distribution
ax2 = fig.add_subplot(gs[0,2])
ax2.hist(error_df['AbsError'], bins=range(0, int(error_df['AbsError'].max())+2),
         color='#4a9eff', alpha=0.8, edgecolor='#0f0f0f')
ax2.axvline(final_results[best_final]['MAE'], color='#ffd700', linewidth=2,
            linestyle='--', label=f'MAE={r_best["MAE"]}')
ax2.set_title('Absolute Error Distribution', color='white', fontsize=10)
ax2.set_xlabel('|Error| (positions)')
ax2.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)
ax2.grid(alpha=0.25)

# [1,0:2] Actual vs Predicted
ax3 = fig.add_subplot(gs[1,0:2])
for i, row in error_df.iterrows():
    c = TEAM_COLORS.get(row['Team'],'#888888')
    ax3.scatter(row['Actual'], row['Predicted'], color=c, s=90, alpha=0.9, zorder=5)
    ax3.annotate(row['Driver'], (row['Actual'], row['Predicted']),
                 textcoords='offset points', xytext=(5,3), fontsize=7.5, color='white')
ax3.plot([1,22],[1,22],'--',color='#444',linewidth=1.2)
ax3.fill_between([1,22],[max(1,l-2) for l in [1,22]],[min(22,l+2) for l in [1,22]],
                 alpha=0.08, color='#4a9eff')
ax3.set_title(f'Actual vs Predicted — {best_final}', color='white', fontsize=10)
ax3.set_xlabel('Actual'); ax3.set_ylabel('Predicted')
ax3.grid(alpha=0.2); ax3.set_xlim(0,23); ax3.set_ylim(-2,26)

# [1,2] Learning curve
ax4 = fig.add_subplot(gs[1,2])
ax4.plot(lc_df['k'], lc_df['Spearman'], color='#00e676', linewidth=2)
ax4.axvline(best_k, color='#ffd700', linewidth=1.5, linestyle='--',
            label=f'Optimal k={best_k}')
ax4.set_xlabel('Feature Count')
ax4.set_title('Learning Curve', color='white', fontsize=10)
ax4.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)
ax4.grid(alpha=0.25)

savefig('09_tuned_dashboard.png'); plt.show()


In [ ]:
# ── Text summary ─────────────────────────────────────────────
print('=' * 65)
print('  🏎️  F1 2026 CHINESE GP — TUNED MODEL SUMMARY')
print('=' * 65)
print()
print('📊 Final Results:')
for name, r in final_results.items():
    marker = '🥇' if name==best_final else '  '
    print(f' {marker} {name:35s}: MAE={r["MAE"]} | Spearman={r["Spearman"]} | '
          f'±2={r["Within±2"]} | ±3={r["Within±3"]}')

print()
print(f'🔑 Key findings:')
print(f'   1. sprint_pos gây multicollinearity với grid_pos')
print(f'      → Set B (no sprint_pos) Linear Reg: Spearman=0.544 > Set A')
print(f'   2. Best α Ridge: {best_alpha} → Spearman={ridge_results[best_alpha]["Spearman"]:.3f}')
print(f'   3. Optimal feature count: k={best_k} (learning curve)')

print()
print('⚠️  Biggest surprises:')
for _, row in error_df.sort_values('AbsError', ascending=False).head(4).iterrows():
    direction = 'better' if row['Error']<0 else 'worse'
    g = f'G{int(row.Grid)}' if pd.notna(row.Grid) else 'G?'
    s = f'→S{int(row.Sprint)}' if pd.notna(row.Sprint) else ''
    print(f'   {row.Driver:5s} ({row.Team}): {g}{s}→R{int(row.Actual)} | '
          f'pred R{row.Predicted:.0f} | {row.AbsError:.1f} pos {direction}')

# ── Export ────────────────────────────────────────────────────
pred_export = error_df[['Driver','Team','Grid','Sprint','Actual','Predicted','Error','AbsError']].copy()
pred_export.columns = ['Driver','Team','Grid_Pos','Sprint_Pos','Actual_Race',
                       'Predicted_Race','Error','AbsError']
pred_export.to_csv(f'{PROC_DIR}/predictions_tuned.csv', index=False)

print()
print(f'✅ Saved: {PROC_DIR}/predictions_tuned.csv')
print()
print('📁 Figures:')
import glob
for f in sorted(glob.glob(f'{FIG_DIR}/*.png')):
    print(f'   {f}')
